# Preprocessing: prepare recordins for training

Purpose
-------

Make raw recordins consistent for training:
- convert to mono, target sample rate (e.g. 22050)
- trim long leading/trailing silence, normalize loudness
- export 16-bit PCM WAVs into data/processed/wavs/
- produce / update data/processed/metadata.csv (LJSpeech style: filename|transcriptions|speaker_id)

Run this after you place files in data/raw/recordings/. Work on a small subset first.

In [ ]:
from pathlib import Path
import soundfile as sf
import numpy as np
from IPython.display import Audio, display
import os

RAW_DIR = Path("data/raw/recordings")
OUTPUT_DIR = Path("data/processed/wavs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SAMPLE_RATE = 22050
MIN_DURATION = 0.2          #seconds, drop short files
MAX_DURATION = 20.0         #seconds, split or drop long files

def list_audio(folder=RAW_DIR, number_of_files=10):
    files = sorted([p for p in Path(folder).rglob("*") if p.suffix.lower() in [".wav", ".flac", ".mp3", ".m4a", ".ogg"]])
    for i,f in enumerate(files[:number_of_files], 1):
        try:
            info = sf.info(str(f))
            duration = info.frames / info.samplerate if info.frames and info.samplerate else None
            print(f"{i}. {f} - sample_rate={info.samplerate}, channels={info.channels}, duration={duration:.3f}s")    
        except Exception as e:
            print(f"{i}. {f} - ERROR: {e}")
    return files

files = list_audio()

#play the first file (if any)
if files:
    print("Playing first file:")
    display(Audio(str(files[0])))


## Processing function

Read -> mono -> resample -> trim silence -> normalize -> save 16-bit WAV

In [ ]:
import math

try:
    import librosa
except Exception:
    librosa = None
    
def to_mono(audio):
    if audio.ndim == 1:
        return audio
    return np.mean(audio, axis=1)

def normalize_rms(wav, target_db = -20.0):
    rms = np.sqrt(np.mean(wav**2))
    if rms <= 0:
        return wav
    target_lin = 10.0 ** (target_db / 20.0)
    scale = target_lin / tms
    return wav * scale

def trim_silence_librosa(wav, sample_rate, top_db = 40):
    if librosa is None:
        return wav #do nothing
    
    intervals = librosa.effects.split(wav, top_db = top_db)
    if intervals.size == 0:
        return wav
    
    start, end = intervals[0,0], intervals[-1,1]
    
    return wav[start:end]

def process_file(in_path: Path, output_dir: Path, target_sample_rate = TARGET_SAMPLE_RATE):
    data, sample_rate = sf.read(str(in_path), always_2d=True)
    
    wav = to_mono()
    
    if sample_rate != target_sample_rate:
        if librosa:
            wav = librosa.resample(wav, orig_sr=sample_rate, target_sr= target_sample_rate)
            sample_rate = target_sample_rate
        else: 
            # simple naive liner-interpolation resample (not ideal)
            ratio = target_sample_rate / sample_rate
            resampled_length = int(round(len(wav) * ratio))
            wav = np.interp(np.linspace(0, len(wav, resampled_length), np.arange(len(wav))),wav)
            
        wav = trim_silence_librosa(wav, sample_rate)
        wav = normalize_rms(wav, target_db = -20.0)
        
        duration = len(wav) / sample_rate
        
        if duration < MIN_DURATION:
            raise ValueError(f"Too short ({duration:.3f}s)")
        if duration > MAX_DURATION:
            raise ValueError(f"Too long ({duration:.3f}s)")
        
        output_path = output_dir / in_path.name
        sf.write(str(output_path), wav, sample_rate, subtype="PCM_16")
        return output_path, sample_rate, duration
    
# test on the first file
if files:
    try:
        output_path, sample_rate, duration = process_file(files[0], OUTPUT_DIR)
        print("Processed ->", output_path, sample_rate, duration)
        display(Audio(str(output_path)))
    except Exception as e:
        print("Processing error:", e)